# Aplicación RAG con NVIDIA NIM en Google Colab (mi versión: Bloque A)

Esta versión es **autosuficiente**: configura NVIDIA NIM, carga los documentos, crea un índice Qdrant temporal y publica una interfaz Gradio. Además incluye la evidencia de las tareas **A1 a A4** del workshop.

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3.5-super-vl-120b-a12b`

> **Cambio respecto al notebook del curso:** el notebook original usa `nvidia/nemotron-3-super-120b-a12b`, pero mi cuenta de NVIDIA no lo tiene habilitado (la API responde 404). Usé `nvidia/nemotron-3.5-super-vl-120b-a12b`, que tiene el mismo tamaño (120B totales, 12B activos) y responde bien en español.

> En Colab, ejecuta las celdas en orden. La última celda mostrará una URL pública temporal de Gradio.

## 1. Instalar dependencias

Después de la instalación no suele ser necesario reiniciar el runtime. Si Colab lo solicita, reinícialo y vuelve a ejecutar desde la celda siguiente.

In [1]:
import sys

# Solo instalamos en Colab. En local las dependencias ya están en el entorno,
# y así evitamos que un `-U` cambie versiones sin querer.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic(
        "pip",
        "install -qU gradio langchain langchain-core langchain-text-splitters "
        "langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv",
    )

## 2. Configurar la API key

En Colab, abre **🔑 Secrets**, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook. En local, la clave se lee de un archivo `.env` (se busca hacia arriba desde la carpeta actual, así funciona desde `notebooks/` o desde la raíz del repo). La clave nunca se imprime.

In [2]:
import os
import re
import json
import time
import difflib
import getpass
from pathlib import Path
from typing import List, Dict

import gradio as gr
from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown, display
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings, ChatNVIDIA
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

# Busca el .env en la carpeta actual o en las carpetas padre (no falla si no existe).
load_dotenv(find_dotenv(usecwd=True))

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

# CAMBIO: el modelo del curso (nvidia/nemotron-3-super-120b-a12b) devuelve 404 en mi
# cuenta. Uso nemotron-3.5-super-vl-120b-a12b, del mismo tamaño (120B / 12B activos).
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"
COLLECTION = "company_docs_gradio"

print("✅ Configuración NVIDIA NIM lista")


def with_retry(function, *args, tries=6, base_wait=2.0, **kwargs):
    """Llama a `function` y reintenta con backoff exponencial ante 429, 5xx o timeouts.

    La clave se comparte con otras personas, así que no hacemos ráfagas.
    """
    for attempt in range(tries):
        try:
            return function(*args, **kwargs)
        except Exception as error:  # noqa: BLE001
            text = f"{type(error).__name__}: {error}"
            transient = any(token in text for token in (
                "429", "500", "502", "503", "504", "Timeout", "timed out", "rate", "Too Many",
            ))
            if not transient or attempt == tries - 1:
                raise
            wait = base_wait * 2 ** attempt
            print(f"⏳ Error transitorio ({text[:70]}...). Reintento en {wait:.0f} s")
            time.sleep(wait)


def md_table(headers, rows):
    """Imprime una tabla Markdown (escapa barras y saltos de línea)."""
    clean = lambda value: str(value).replace("|", "\\|").replace("\n", " ").strip()
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(clean(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))

/Users/juandavidsierraorozco/.venvs/rag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Configuración NVIDIA NIM lista


## 3. Cargar los documentos

El corpus propio vive en `data/company_documents.json` (campos `id`, `title`, `content` y `metadata`). La ruta se busca en varios lugares relativos, para que funcione igual en local (desde `notebooks/` o desde la raíz del repo) y en Colab (si subes el archivo a la raíz de la sesión o a una carpeta `data/`). Si no existe, la aplicación usa un conjunto de demostración.

In [3]:
DEMO_DOCUMENTS = [
    {
        "id": "product_001", "title": "CloudSync Pro — Precios",
        "content": "El plan Enterprise cuesta $49/mes por usuario con compromiso anual. El plan mensual sin compromiso cuesta $59/mes por usuario.",
        "metadata": {"department": "sales", "type": "pricing"},
    },
    {
        "id": "hr_001", "title": "Política de trabajo remoto",
        "content": "Los empleados pueden trabajar de forma remota hasta 3 días por semana, con aprobación del jefe directo.",
        "metadata": {"department": "hr", "type": "policy"},
    },
    {
        "id": "finance_001", "title": "Política de reembolsos",
        "content": "Los reembolsos mayores a $100 requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original.",
        "metadata": {"department": "finance", "type": "policy"},
    },
    {
        "id": "hr_002", "title": "Onboarding",
        "content": "Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa.",
        "metadata": {"department": "hr", "type": "procedure"},
    },
]

# Rutas candidatas, todas relativas: raíz del repo, carpeta notebooks/ o raíz de Colab.
DATA_CANDIDATES = [
    Path("data/company_documents.json"),
    Path("../data/company_documents.json"),
    Path("company_documents.json"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), DATA_CANDIDATES[0])


def load_documents(path: Path) -> List[Dict]:
    if not path.exists():
        print(f"ℹ️ No se encontró {path}; se usarán documentos de demostración.")
        return DEMO_DOCUMENTS

    with path.open(encoding="utf-8") as file:
        documents = json.load(file)

    required = {"id", "title", "content"}
    for index, document in enumerate(documents):
        missing = required - document.keys()
        if missing:
            raise ValueError(f"Al documento {index} le faltan campos: {missing}")
        document.setdefault("metadata", {})
    return documents


company_documents = load_documents(DATA_PATH)
print(f"✅ {len(company_documents)} documentos disponibles desde {DATA_PATH}")

✅ 11 documentos disponibles desde ../data/company_documents.json


## 4. Construir el pipeline RAG

Esta celda crea los fragmentos, calcula embeddings con NVIDIA NIM y los indexa en Qdrant. El índice vive en memoria mientras la sesión esté activa.

Cambios respecto al original:
- **A3:** el system prompt se separa en dos constantes (`SYSTEM_PROMPT_ANTES` es el del curso y `SYSTEM_PROMPT_DESPUES` es el nuevo) y `build_pipeline` recibe el prompt como parámetro. El valor por defecto es el nuevo.
- **A4:** `build_pipeline` devuelve también el `vectorstore`.
- **A2:** `chunk_size` y `chunk_overlap` son parámetros (más abajo explico por qué uso 700 en mi corpus).

In [4]:
SYSTEM_PROMPT_ANTES = (
    "Eres un asistente empresarial. Responde únicamente con información "
    "del contexto proporcionado. Si el contexto no contiene la respuesta, "
    "indica que no tienes esa información. Responde en español y de forma concisa."
)

SYSTEM_PROMPT_DESPUES = (
    "Eres un asistente empresarial. Responde únicamente con información "
    "del contexto proporcionado. Cita siempre el título del documento entre "
    "corchetes, por ejemplo [Política de reembolsos]. Si el contexto no alcanza, "
    "responde exactamente: «No encuentro esa información en los documentos.» "
    "Responde en español, en máximo 3 oraciones."
)
FRASE_FIJA = "No encuentro esa información en los documentos."

CHUNK_SIZE = 700      # ver la justificación en la sección A2
CHUNK_OVERLAP = 70    # 10 % del tamaño del chunk
SCORE_THRESHOLD = 0.3
# OJO (descubierto en A4): en el retriever de langchain-qdrant con distancia coseno el
# `score_threshold` NO está en escala coseno, sino en relevancia = (coseno + 1) / 2.
# Por eso 0.3 equivale a coseno >= -0.4 y en la práctica no filtra nada.


def build_pipeline(
    documents: List[Dict],
    system_prompt: str = SYSTEM_PROMPT_DESPUES,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    score_threshold: float = SCORE_THRESHOLD,
):
    embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
    llm = ChatNVIDIA(
        model=GENERATION_MODEL,
        temperature=0.2,
        top_p=0.95,
        max_completion_tokens=700,
    )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks: List[Document] = []
    for document in documents:
        for index, text in enumerate(splitter.split_text(document["content"])):
            chunks.append(Document(
                page_content=text,
                metadata={
                    "source_id": document["id"],
                    "title": document["title"],
                    "chunk_index": index,
                    **document.get("metadata", {}),
                },
            ))

    # La dimensión se obtiene del modelo y no queda codificada manualmente.
    embedding_dim = len(embeddings.embed_query("prueba de dimensión"))
    qdrant_client = QdrantClient(":memory:")
    qdrant_client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=embedding_dim, distance=Distance.COSINE),
    )

    vectorstore = QdrantVectorStore(
        client=qdrant_client,
        collection_name=COLLECTION,
        embedding=embeddings,
    )
    vectorstore.add_documents(chunks)

    retriever = vectorstore.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": score_threshold},
    )

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "CONTEXTO:\n{context}\n\nPREGUNTA:\n{question}"),
    ])

    def format_docs(items: List[Document]) -> str:
        return "\n\n---\n\n".join(
            f"[{item.metadata.get('title', 'Sin título')}]\n{item.page_content}"
            for item in items
        )

    setup_and_retrieval = RunnableParallel(
        question=RunnablePassthrough(),
        documents=retriever,
    )
    answer_from_documents = (
        {
            "context": lambda item: format_docs(item["documents"]),
            "question": lambda item: item["question"],
        }
        | prompt
        | llm
        | StrOutputParser()
    )
    pipeline = setup_and_retrieval.assign(answer=answer_from_documents)
    return pipeline, chunks, vectorstore


def ask(pipeline, question: str, pause: float = 1.5):
    """Pregunta al pipeline con reintentos y una pausa corta (la clave es compartida)."""
    result = with_retry(pipeline.invoke, question)
    time.sleep(pause)
    titles = []
    for document in result["documents"]:
        title = document.metadata.get("title", "Sin título")
        if title not in titles:
            titles.append(title)
    return result["answer"].strip(), titles


print("✅ build_pipeline y utilidades definidas")

✅ build_pipeline y utilidades definidas


## 5. Evidencia del Bloque A

### A1 · Ejecuta la app RAG de punta a punta

Primero ejecuto el pipeline **tal como viene en el curso** (documentos de demostración y `SYSTEM_PROMPT_ANTES`), con las 4 preguntas de ejemplo y una quinta pregunta cuya respuesta no está en los documentos.

In [5]:
print("Construyendo el índice con los documentos de demostración...")
demo_pipeline, demo_chunks, demo_vectorstore = with_retry(
    build_pipeline, DEMO_DOCUMENTS, system_prompt=SYSTEM_PROMPT_ANTES
)
print(f"✅ Pipeline listo: {len(DEMO_DOCUMENTS)} documentos, {len(demo_chunks)} chunks")

A1_QUESTIONS = [
    "¿Cuál es el precio del plan Enterprise de CloudSync Pro?",
    "¿Cuántos días pueden trabajar remotamente los empleados?",
    "¿Qué pasa con los reembolsos mayores a $100?",
    "¿Qué ocurre durante la primera semana de onboarding?",
    "¿Cuál es la política de vacaciones de la empresa?",  # NO está en los documentos
]

a1_rows = []
for number, question in enumerate(A1_QUESTIONS, start=1):
    answer, titles = ask(demo_pipeline, question)
    a1_rows.append([number, question, answer, "; ".join(titles) or "(ninguna)"])

md_table(["#", "Pregunta", "Respuesta", "Fuentes recuperadas"], a1_rows)

Construyendo el índice con los documentos de demostración...


/Users/juandavidsierraorozco/.venvs/rag/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo: 4 documentos, 4 chunks


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 2 s


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 4 s


| # | Pregunta | Respuesta | Fuentes recuperadas |
|---|---|---|---|
| 1 | ¿Cuál es el precio del plan Enterprise de CloudSync Pro? | $49/mes por usuario con compromiso anual. | CloudSync Pro — Precios; Onboarding; Política de reembolsos |
| 2 | ¿Cuántos días pueden trabajar remotamente los empleados? | Hasta 3 días por semana, con aprobación del jefe directo. | Política de trabajo remoto; Onboarding; Política de reembolsos |
| 3 | ¿Qué pasa con los reembolsos mayores a $100? | Requieren aprobación del supervisor. Una vez aprobados, se procesan en 3 a 5 días hábiles al método de pago original. | Política de reembolsos; Política de trabajo remoto; CloudSync Pro — Precios |
| 4 | ¿Qué ocurre durante la primera semana de onboarding? | Durante la primera semana se completan módulos obligatorios de seguridad, cumplimiento y cultura de la empresa. | Onboarding; Política de trabajo remoto; Política de reembolsos |
| 5 | ¿Cuál es la política de vacaciones de la empresa? | No tengo información sobre la política de vacaciones de la empresa en el contexto proporcionado. | Política de trabajo remoto; Onboarding; Política de reembolsos |

### A2 · Tu propio corpus de documentos

Corpus: **NodoIA**, un semillero de IA de EAFIT *inventado* (11 documentos con reglamento, horarios, proyectos, presupuesto, convocatoria, evento, recursos, certificados y contacto). Cada documento tiene `metadata` con dos claves: `department` y `type`.

**Sobre `chunk_size`:** el original usa 400 caracteres. Mis documentos miden entre ~360 y ~590 caracteres, así que con 400 casi todos se partirían en dos pedazos, y el segundo pedazo perdería el contexto de qué documento habla (el título va en la metadata, no en el texto que se embebe). Cada documento es una unidad de significado completa, por lo que prefiero que quede en un solo chunk: subo `chunk_size` a 700 y `chunk_overlap` a 70 (10 %). La celda siguiente cuenta los chunks con cada tamaño (solo se parte texto, sin llamar a la API).

In [6]:
for size, overlap in [(400, 40), (700, 70)]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap, separators=["\n\n", "\n", ".", " "]
    )
    per_doc = [len(splitter.split_text(doc["content"])) for doc in company_documents]
    print(f"chunk_size={size:>3}, overlap={overlap:>2}: {sum(per_doc)} chunks "
          f"para {len(company_documents)} documentos (docs partidos: {sum(n > 1 for n in per_doc)})")

lengths = [len(doc["content"]) for doc in company_documents]
print(f"Largo de los documentos: min={min(lengths)}, max={max(lengths)} caracteres")

chunk_size=400, overlap=40: 18 chunks para 11 documentos (docs partidos: 7)
chunk_size=700, overlap=70: 11 chunks para 11 documentos (docs partidos: 0)
Largo de los documentos: min=361, max=591 caracteres


In [7]:
print("Construyendo el índice con mi corpus (prompt original)...")
pipe_antes, chunks_antes, vs_antes = with_retry(
    build_pipeline, company_documents, system_prompt=SYSTEM_PROMPT_ANTES
)
print(f"✅ Pipeline listo: {len(company_documents)} documentos, {len(chunks_antes)} chunks (chunk_size={CHUNK_SIZE})")

A2_QUESTIONS = [
    ("sí", "¿Qué promedio acumulado mínimo necesito para entrar al semillero?"),
    ("sí", "¿Qué día y a qué hora es la reunión general?"),
    ("sí", "¿Qué porcentaje del presupuesto de 2026 se destina a cómputo en la nube?"),
    ("sí", "¿Cuándo es el hackatón y cuánto gana el primer puesto?"),
    ("sí", "¿Cuántas horas seguidas puedo reservar la GPU?"),
    ("no", "¿Cuánto cuesta la matrícula de la maestría en IA de EAFIT?"),
    ("no", "¿Quién ganó el Mundial de fútbol de 2022?"),
]

a2_results = {}
a2_rows = []
for number, (kind, question) in enumerate(A2_QUESTIONS, start=1):
    answer, titles = ask(pipe_antes, question)
    a2_results[question] = (answer, titles)
    a2_rows.append([number, kind, question, answer, "; ".join(titles) or "(ninguna)"])

md_table(["#", "¿Está en el corpus?", "Pregunta", "Respuesta", "Fuentes recuperadas"], a2_rows)

Construyendo el índice con mi corpus (prompt original)...


/Users/juandavidsierraorozco/.venvs/rag/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo: 11 documentos, 11 chunks (chunk_size=700)


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 2 s


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 4 s


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 2 s


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 4 s


| # | ¿Está en el corpus? | Pregunta | Respuesta | Fuentes recuperadas |
|---|---|---|---|---|
| 1 | sí | ¿Qué promedio acumulado mínimo necesito para entrar al semillero? | Necesitas un promedio acumulado mínimo de 3,5. | Reglamento de membresía; Recursos de cómputo; Presentación del semillero NodoIA |
| 2 | sí | ¿Qué día y a qué hora es la reunión general? | La reunión general es los martes de 6:00 p. m. a 8:00 p. m. | Horario de reuniones; Reglamento de membresía; Canales de comunicación |
| 3 | sí | ¿Qué porcentaje del presupuesto de 2026 se destina a cómputo en la nube? | El 40 % del presupuesto de 2026 se destina a cómputo en la nube. | Presupuesto anual 2026; Hackatón NodoIA Day; Convocatoria de ingreso 2026-2 |
| 4 | sí | ¿Cuándo es el hackatón y cuánto gana el primer puesto? | El hackatón es el 14 de noviembre de 2026 y el primer puesto gana 1.500.000 pesos. | Hackatón NodoIA Day; Horario de reuniones; Recursos de cómputo |
| 5 | sí | ¿Cuántas horas seguidas puedo reservar la GPU? | Hasta 8 horas continuas. | Recursos de cómputo; Certificados y horas de semillero; Reglamento de membresía |
| 6 | no | ¿Cuánto cuesta la matrícula de la maestría en IA de EAFIT? | No tengo esa información. El contexto proporcionado solo incluye detalles sobre el semillero NodoIA, su hackatón y su reglamento de membresía, pero no menciona la maestría en IA ni el costo de su matrícula. | Presentación del semillero NodoIA; Hackatón NodoIA Day; Reglamento de membresía |
| 7 | no | ¿Quién ganó el Mundial de fútbol de 2022? | No tengo esa información en el contexto proporcionado. | Hackatón NodoIA Day; Convocatoria de ingreso 2026-2; Proyecto Chatbot de Bienestar |

### A3 · Ingeniería del prompt (antes y después)

Tres preguntas de mi tabla A2: una **fácil** (la respuesta está en un solo documento), una **ambigua** (varios documentos podrían aplicar) y una **fuera de contexto**. Primero se guardan las respuestas con el prompt original (pipeline `pipe_antes`, ya construido).

In [8]:
A3_QUESTIONS = [
    ("fácil", "¿Cuántas horas seguidas puedo reservar la GPU?"),
    ("ambigua", "¿Cuándo es la reunión?"),
    ("fuera de contexto", "¿Quién ganó el Mundial de fútbol de 2022?"),
]

a3_antes = {}
for kind, question in A3_QUESTIONS:
    a3_antes[question] = ask(pipe_antes, question)

for kind, question in A3_QUESTIONS:
    answer, titles = a3_antes[question]
    print(f"[ANTES · {kind}] {question}\n  -> {answer}\n  fuentes: {titles}\n")

[ANTES · fácil] ¿Cuántas horas seguidas puedo reservar la GPU?
  -> Hasta 8 horas continuas.
  fuentes: ['Recursos de cómputo', 'Certificados y horas de semillero', 'Reglamento de membresía']

[ANTES · ambigua] ¿Cuándo es la reunión?
  -> La reunión general de NodoIA es los martes de 6:00 p. m. a 8:00 p. m. en el bloque 19, sala 302.
  fuentes: ['Horario de reuniones', 'Canales de comunicación', 'Hackatón NodoIA Day']

[ANTES · fuera de contexto] ¿Quién ganó el Mundial de fútbol de 2022?
  -> No tengo esa información en el contexto proporcionado.
  fuentes: ['Hackatón NodoIA Day', 'Convocatoria de ingreso 2026-2', 'Proyecto Chatbot de Bienestar']



In [9]:
# Diff del system prompt (una oración por línea para que se lea bien).
split_sentences = lambda text: text.replace(". ", ".\n").replace("» ", "»\n").splitlines()
for line in difflib.unified_diff(
    split_sentences(SYSTEM_PROMPT_ANTES), split_sentences(SYSTEM_PROMPT_DESPUES),
    fromfile="prompt_antes", tofile="prompt_despues", lineterm="", n=0,
):
    print(line)

--- prompt_antes
+++ prompt_despues
@@ -3,2 +3,3 @@
-Si el contexto no contiene la respuesta, indica que no tienes esa información.
-Responde en español y de forma concisa.
+Cita siempre el título del documento entre corchetes, por ejemplo [Política de reembolsos].
+Si el contexto no alcanza, responde exactamente: «No encuentro esa información en los documentos.»
+Responde en español, en máximo 3 oraciones.


In [10]:
# Reconstruyo el pipeline con el prompt nuevo. Este será el pipeline final de la app.
print("Reconstruyendo el pipeline con el prompt nuevo...")
rag_pipeline, indexed_chunks, vectorstore = with_retry(build_pipeline, company_documents)
print(f"✅ Pipeline listo: {len(company_documents)} documentos, {len(indexed_chunks)} chunks")

a3_despues = {}
for kind, question in A3_QUESTIONS:
    a3_despues[question] = ask(rag_pipeline, question)

Reconstruyendo el pipeline con el prompt nuevo...


/Users/juandavidsierraorozco/.venvs/rag/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo: 11 documentos, 11 chunks


In [11]:
TITLES = [doc["title"] for doc in company_documents]


def count_sentences(text: str) -> int:
    text = re.sub(r"\b([ap])\. m\.", r"\1m", text)  # "6:00 p. m." no es el fin de una oración
    return len([part for part in re.split(r"(?<=[.!?»])\s+", text.strip()) if part])


def cites_title(text: str) -> bool:
    found = re.findall(r"\[([^\]]+)\]", text)
    return any(any(t.lower() in f.lower() or f.lower() in t.lower() for t in TITLES) for f in found)


rows, metrics = [], []
for kind, question in A3_QUESTIONS:
    before, _ = a3_antes[question]
    after, _ = a3_despues[question]
    row = {
        "kind": kind,
        "cita_antes": cites_title(before), "cita_despues": cites_title(after),
        "frase_antes": FRASE_FIJA in before, "frase_despues": FRASE_FIJA in after,
        "oraciones_antes": count_sentences(before), "oraciones_despues": count_sentences(after),
    }
    metrics.append(row)
    rows.append([kind, question, before, after])

md_table(["Tipo", "Pregunta", "ANTES (prompt original)", "DESPUÉS (prompt nuevo)"], rows)

yes = lambda flag: "sí" if flag else "no"
md_table(
    ["Tipo", "Cita [título] antes", "Cita [título] después", "Frase fija antes", "Frase fija después",
     "Oraciones antes", "Oraciones después"],
    [[m["kind"], yes(m["cita_antes"]), yes(m["cita_despues"]), yes(m["frase_antes"]),
      yes(m["frase_despues"]), m["oraciones_antes"], m["oraciones_despues"]] for m in metrics],
)

| Tipo | Pregunta | ANTES (prompt original) | DESPUÉS (prompt nuevo) |
|---|---|---|---|
| fácil | ¿Cuántas horas seguidas puedo reservar la GPU? | Hasta 8 horas continuas. | Puedes reservar la GPU por un máximo de 8 horas continuas [Recursos de cómputo]. |
| ambigua | ¿Cuándo es la reunión? | La reunión general de NodoIA es los martes de 6:00 p. m. a 8:00 p. m. en el bloque 19, sala 302. | La reunión general de NodoIA es los martes de 6:00 p. m. a 8:00 p. m. en el bloque 19, sala 302 [Horario de reuniones]. Los jueves de 4:00 p. m. a 6:00 p. m. hay sesión de laboratorio práctico en la sala de cómputo [Horario de reuniones]. Durante las semanas de parciales las reuniones se suspenden y se avisa por Discord [Horario de reuniones]. |
| fuera de contexto | ¿Quién ganó el Mundial de fútbol de 2022? | No tengo esa información en el contexto proporcionado. | No encuentro esa información en los documentos. |

| Tipo | Cita [título] antes | Cita [título] después | Frase fija antes | Frase fija después | Oraciones antes | Oraciones después |
|---|---|---|---|---|---|---|
| fácil | no | sí | no | no | 1 | 1 |
| ambigua | no | sí | no | no | 1 | 3 |
| fuera de contexto | no | no | no | sí | 1 | 1 |

### A4 · Muestra el score de similitud en las fuentes

`similarity_search_with_score(question, k=3)` devuelve los 3 chunks más parecidos **sin aplicar el umbral**, con su score de coseno. Pruebo 5 preguntas (3 buenas y 2 fuera de contexto) y miro cómo se separan los scores.

In [12]:
A4_QUESTIONS = [
    ("buena", "¿Cuántos cupos se ofrecieron en la convocatoria 2026-2?"),
    ("buena", "¿Qué pasa si falto a tres reuniones seguidas?"),
    ("buena", "¿Cuántos integrantes tiene el proyecto de plagas en café?"),
    ("fuera", "¿Cuál es el horario de la biblioteca de EAFIT?"),
    ("fuera", "¿Cuál es la receta de la bandeja paisa?"),
]

a4_data = []
for kind, question in A4_QUESTIONS:
    scored = with_retry(vectorstore.similarity_search_with_score, question, k=3)
    time.sleep(1.0)
    a4_data.append((kind, question, [(d.metadata["title"], s) for d, s in scored]))

rows = []
for kind, question, scored in a4_data:
    rows.append([kind, question] + [f"{title} ({score:.3f})" for title, score in scored])
md_table(["Tipo", "Pregunta", "Fuente 1 (score)", "Fuente 2 (score)", "Fuente 3 (score)"], rows)

good = [scored[0][1] for kind, _, scored in a4_data if kind == "buena"]
bad = [scored[0][1] for kind, _, scored in a4_data if kind == "fuera"]
print(f"Score top-1 de preguntas buenas: min={min(good):.3f}  max={max(good):.3f}")
print(f"Score top-1 de preguntas fuera de contexto: min={min(bad):.3f}  max={max(bad):.3f}")
print(f"Brecha entre el peor acierto y el mejor fuera de contexto: {min(good) - max(bad):.3f}")

| Tipo | Pregunta | Fuente 1 (score) | Fuente 2 (score) | Fuente 3 (score) |
|---|---|---|---|---|
| buena | ¿Cuántos cupos se ofrecieron en la convocatoria 2026-2? | Convocatoria de ingreso 2026-2 (0.516) | Hackatón NodoIA Day (0.217) | Presupuesto anual 2026 (0.194) |
| buena | ¿Qué pasa si falto a tres reuniones seguidas? | Reglamento de membresía (0.354) | Horario de reuniones (0.252) | Certificados y horas de semillero (0.129) |
| buena | ¿Cuántos integrantes tiene el proyecto de plagas en café? | Proyecto Detector de Plagas en Café (0.524) | Presentación del semillero NodoIA (0.192) | Reglamento de membresía (0.159) |
| fuera | ¿Cuál es el horario de la biblioteca de EAFIT? | Reglamento de membresía (0.245) | Presentación del semillero NodoIA (0.224) | Hackatón NodoIA Day (0.206) |
| fuera | ¿Cuál es la receta de la bandeja paisa? | Presupuesto anual 2026 (0.070) | Hackatón NodoIA Day (0.063) | Presentación del semillero NodoIA (0.057) |

Score top-1 de preguntas buenas: min=0.354  max=0.524
Score top-1 de preguntas fuera de contexto: min=0.070  max=0.245
Brecha entre el peor acierto y el mejor fuera de contexto: 0.109


In [13]:
# Datos adicionales para el umbral: el top-1 de las 7 preguntas de A2 (solo embeddings).
extra = []
for kind, question in A2_QUESTIONS:
    scored = with_retry(vectorstore.similarity_search_with_score, question, k=3)
    time.sleep(1.0)
    extra.append((kind, question, scored[0][0].metadata["title"], scored[0][1]))

md_table(["¿Está en el corpus?", "Pregunta", "Top-1", "Score"],
         [[k, q, t, f"{s:.3f}"] for k, q, t, s in extra])

all_good = good + [s for k, _, _, s in extra if k == "sí"]
all_bad = bad + [s for k, _, _, s in extra if k == "no"]
print(f"Con las {len(all_good)} preguntas que sí están: min top-1 = {min(all_good):.3f}")
print(f"Con las {len(all_bad)} preguntas que no están: max top-1 = {max(all_bad):.3f}")
print(f"Punto medio de la brecha = {(min(all_good) + max(all_bad)) / 2:.3f}")

# Qué pasaría con distintos umbrales de coseno (sin llamar a la API).
for threshold in (0.3, 0.4, 0.5, 0.55, 0.6, 0.65):
    kept_good = sum(score >= threshold for score in all_good)
    kept_bad = sum(score >= threshold for score in all_bad)
    print(f"umbral={threshold:.2f}: preguntas buenas que conservan contexto={kept_good}/{len(all_good)}, "
          f"fuera de contexto que aún lo reciben={kept_bad}/{len(all_bad)}")

| ¿Está en el corpus? | Pregunta | Top-1 | Score |
|---|---|---|---|
| sí | ¿Qué promedio acumulado mínimo necesito para entrar al semillero? | Reglamento de membresía | 0.437 |
| sí | ¿Qué día y a qué hora es la reunión general? | Horario de reuniones | 0.388 |
| sí | ¿Qué porcentaje del presupuesto de 2026 se destina a cómputo en la nube? | Presupuesto anual 2026 | 0.570 |
| sí | ¿Cuándo es el hackatón y cuánto gana el primer puesto? | Hackatón NodoIA Day | 0.532 |
| sí | ¿Cuántas horas seguidas puedo reservar la GPU? | Recursos de cómputo | 0.564 |
| no | ¿Cuánto cuesta la matrícula de la maestría en IA de EAFIT? | Presentación del semillero NodoIA | 0.393 |
| no | ¿Quién ganó el Mundial de fútbol de 2022? | Hackatón NodoIA Day | 0.114 |

Con las 8 preguntas que sí están: min top-1 = 0.354
Con las 4 preguntas que no están: max top-1 = 0.393
Punto medio de la brecha = 0.374
umbral=0.30: preguntas buenas que conservan contexto=8/8, fuera de contexto que aún lo reciben=1/4
umbral=0.40: preguntas buenas que conservan contexto=6/8, fuera de contexto que aún lo reciben=0/4
umbral=0.50: preguntas buenas que conservan contexto=5/8, fuera de contexto que aún lo reciben=0/4
umbral=0.55: preguntas buenas que conservan contexto=2/8, fuera de contexto que aún lo reciben=0/4
umbral=0.60: preguntas buenas que conservan contexto=0/8, fuera de contexto que aún lo reciben=0/4
umbral=0.65: preguntas buenas que conservan contexto=0/8, fuera de contexto que aún lo reciben=0/4



#### Un hallazgo sobre la escala del umbral

En las tablas de A1 y A2 las preguntas fuera de contexto recuperan igual 3 fuentes, aunque el retriever tiene `score_threshold=0.3`. La causa: `QdrantVectorStore` convierte el coseno en una *relevancia* con `(coseno + 1) / 2` antes de comparar con el umbral. Con 0.3 el filtro equivale a coseno >= -0.4, así que nunca descarta nada. La celda siguiente lo comprueba con la misma consulta.

In [14]:

def cos_to_relevance(cosine: float) -> float:
    """Escala que usa el retriever de langchain-qdrant para el score_threshold."""
    return (cosine + 1) / 2


probe = "¿Quién ganó el Mundial de fútbol de 2022?"
cosine_scores = with_retry(vectorstore.similarity_search_with_score, probe, k=3)
relevance_scores = with_retry(vectorstore.similarity_search_with_relevance_scores, probe, k=3)
for (doc, cosine), (_, relevance) in zip(cosine_scores, relevance_scores):
    print(f"{doc.metadata['title']:<36} coseno={cosine:.3f}  relevancia={relevance:.3f}  "
          f"(coseno+1)/2={cos_to_relevance(cosine):.3f}")
print("Con score_threshold=0.3 todas pasan: 0.3 se compara con la relevancia, no con el coseno.")


Hackatón NodoIA Day                  coseno=0.114  relevancia=0.557  (coseno+1)/2=0.557
Convocatoria de ingreso 2026-2       coseno=0.090  relevancia=0.545  (coseno+1)/2=0.545
Proyecto Chatbot de Bienestar        coseno=0.068  relevancia=0.534  (coseno+1)/2=0.534
Con score_threshold=0.3 todas pasan: 0.3 se compara con la relevancia, no con el coseno.



#### Aplicar el umbral propuesto

Propongo un umbral de **coseno >= 0.30**, que en la escala del retriever es `score_threshold = (0.30 + 1) / 2 = 0.65`. La justificación con los datos está arriba y en `RESPUESTAS.md`. Reconstruyo el pipeline final con ese valor y repito las preguntas fuera de contexto y las dos buenas más cercanas al borde.

In [15]:

PROPOSED_COSINE_THRESHOLD = 0.30

print("Reconstruyendo el pipeline final con el umbral propuesto...")
rag_pipeline, indexed_chunks, vectorstore = with_retry(
    build_pipeline, company_documents, score_threshold=cos_to_relevance(PROPOSED_COSINE_THRESHOLD)
)
print(f"✅ Pipeline listo con score_threshold={cos_to_relevance(PROPOSED_COSINE_THRESHOLD):.2f} "
      f"(coseno >= {PROPOSED_COSINE_THRESHOLD})")

CHECK_QUESTIONS = [
    ("fuera", "¿Quién ganó el Mundial de fútbol de 2022?"),
    ("fuera", "¿Cuál es la receta de la bandeja paisa?"),
    ("fuera", "¿Cuál es el horario de la biblioteca de EAFIT?"),
    ("fuera", "¿Cuánto cuesta la matrícula de la maestría en IA de EAFIT?"),
    ("buena", "¿Qué pasa si falto a tres reuniones seguidas?"),
    ("buena", "¿Qué día y a qué hora es la reunión general?"),
]
rows = []
for kind, question in CHECK_QUESTIONS:
    answer, titles = ask(rag_pipeline, question)
    rows.append([kind, question, "; ".join(titles) or "(ninguna)", answer])
md_table(["Tipo", "Pregunta", "Fuentes que llegan al LLM", "Respuesta"], rows)


Reconstruyendo el pipeline final con el umbral propuesto...


/Users/juandavidsierraorozco/.venvs/rag/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ Pipeline listo con score_threshold=0.65 (coseno >= 0.3)


No relevant docs were retrieved using the relevance score threshold 0.65


No relevant docs were retrieved using the relevance score threshold 0.65


...

⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 2 s


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 4 s


⏳ Error transitorio (ReadTimeout: HTTPSConnectionPool(host='integrate.api.nvidia.com', port...). Reintento en 2 s


⏳ Error transitorio (Exception: [500] Failed to generate completions
{'message': 'Failed to...). Reintento en 4 s


| Tipo | Pregunta | Fuentes que llegan al LLM | Respuesta |
|---|---|---|---|
| fuera | ¿Quién ganó el Mundial de fútbol de 2022? | (ninguna) | No encuentro esa información en los documentos. |
| fuera | ¿Cuál es la receta de la bandeja paisa? | (ninguna) | No encuentro esa información en los documentos. |
| fuera | ¿Cuál es el horario de la biblioteca de EAFIT? | (ninguna) | No encuentro esa información en los documentos. |
| fuera | ¿Cuánto cuesta la matrícula de la maestría en IA de EAFIT? | Presentación del semillero NodoIA; Hackatón NodoIA Day | No encuentro esa información en los documentos. |
| buena | ¿Qué pasa si falto a tres reuniones seguidas? | Reglamento de membresía | No encuentro esa información en los documentos. |
| buena | ¿Qué día y a qué hora es la reunión general? | Horario de reuniones | La reunión general es los martes de 6:00 p. m. a 8:00 p. m. [Horario de reuniones] |

## 6. Crear la interfaz Gradio

`make_chat` crea la función `chat()` de la interfaz. Igual que en A4, además de la respuesta muestra **cada fuente con su score coseno** (consulta sin filtrar con `similarity_search_with_score(question, k=3)`) y marca cuáles pasan el umbral del retriever.

In [16]:
def make_chat(pipeline, store, show_scores: bool = True, threshold: float = SCORE_THRESHOLD):
    def chat(question: str, history: list | None):
        history = history or []
        question = (question or "").strip()
        if not question:
            return "", history, "Escribe una pregunta."

        try:
            result = with_retry(pipeline.invoke, question)
            answer = result["answer"].strip()
            documents = result["documents"]

            if show_scores:
                scored = with_retry(store.similarity_search_with_score, question, k=3)
                sources_md = (
                    f"**Fuentes recuperadas (score coseno, umbral >= {threshold}):**\n" + "\n".join(
                        f"- {doc.metadata.get('title', 'Sin título')} — {score:.3f} "
                        f"{'✅ usada' if score >= threshold else '✖ bajo el umbral'}"
                        for doc, score in scored
                    )
                )
            else:
                unique_sources = []
                for document in documents:
                    title = document.metadata.get("title", "Sin título")
                    if title not in unique_sources:
                        unique_sources.append(title)
                sources_md = "**Fuentes recuperadas:**\n" + "\n".join(
                    f"- {title}" for title in unique_sources
                ) if unique_sources else "No se encontraron documentos relevantes."

        except Exception as error:
            answer = "Ocurrió un error al consultar el pipeline. Revisa el detalle mostrado debajo."
            sources_md = f"**Error:** `{type(error).__name__}: {error}`"

        history = history + [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]
        return "", history, sources_md

    return chat


def clear_chat():
    return [], ""


def build_demo(pipeline, store, documents, chunks, examples, show_scores=True, threshold=SCORE_THRESHOLD):
    chat = make_chat(pipeline, store, show_scores=show_scores, threshold=threshold)
    with gr.Blocks(title="RAG empresarial con NVIDIA NIM") as demo:
        gr.Markdown(
            "# RAG empresarial\n"
            "Consulta documentos internos mediante **LangChain + NVIDIA NIM + Qdrant**."
        )

        with gr.Row():
            with gr.Column(scale=1):
                gr.Markdown("### Documentos indexados")
                gr.Markdown("\n".join(f"- {doc['title']}" for doc in documents))
                gr.Markdown(f"*{len(chunks)} chunks en el índice*")
                gr.Markdown("---")
                gr.Markdown(
                    f"**Embeddings:** `{EMBEDDING_MODEL}`  \n"
                    f"**LLM:** `{GENERATION_MODEL}`  \n"
                    "**Vector store:** Qdrant en memoria  \n"
                    + (f"**Retrieval:** k=3, umbral coseno >= {threshold}" if show_scores
                       else f"**Retrieval:** k=3, threshold={threshold}")
                )

            with gr.Column(scale=2):
                # Gradio 6 usa mensajes {role, content} por defecto;
                # el antiguo argumento type="messages" ya no existe.
                chatbot = gr.Chatbot(height=430, label="Chat")
                with gr.Row():
                    question_box = gr.Textbox(
                        placeholder="Haz una pregunta sobre los documentos...",
                        label="Pregunta",
                        lines=2,
                        scale=4,
                    )
                    send_button = gr.Button("Enviar", variant="primary", scale=1)

                sources_box = gr.Markdown()
                clear_button = gr.Button("Limpiar", variant="secondary")

        gr.Examples(examples=[[q] for q in examples], inputs=question_box, label="Preguntas de ejemplo")

        send_button.click(chat, inputs=[question_box, chatbot], outputs=[question_box, chatbot, sources_box])
        question_box.submit(chat, inputs=[question_box, chatbot], outputs=[question_box, chatbot, sources_box])
        clear_button.click(clear_chat, outputs=[chatbot, sources_box])
    return demo


demo = build_demo(
    rag_pipeline, vectorstore, company_documents, indexed_chunks,
    examples=[question for kind, question in A2_QUESTIONS if kind == "sí"],
    threshold=PROPOSED_COSINE_THRESHOLD,
)
print("✅ Interfaz lista")

✅ Interfaz lista


## 6. Ejecutar la aplicación en Colab

`share=True` crea el enlace público temporal. La aplicación estará disponible mientras esta celda y el runtime de Colab continúen activos.

In [17]:
# En Colab: URL pública temporal y la celda queda ejecutándose.
# En local: sirve en http://127.0.0.1:7860 (o el siguiente puerto libre) sin bloquear el notebook.
demo.queue().launch(
    share=IN_COLAB,
    debug=IN_COLAB,
    show_error=True,
    prevent_thread_lock=not IN_COLAB,
)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### Detener o volver a ejecutar

- Interrumpe la última celda para detener el servidor.
- Antes de volver a ejecutarla, puedes cerrar instancias previas con `gr.close_all()`.
- El enlace público cambia cada vez que se inicia la aplicación.

In [18]:
# Ejecuta esta celda solo cuando quieras cerrar servidores Gradio anteriores.
gr.close_all()

Closing server running on port: 7860
